In [1]:
#Install XGBoost if not installed
!pip install xgboost

#Import necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

In [2]:
#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")

#Fill missing values in InventoryRatio with median
train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
test_df['InventoryRatio'].fillna(test_df['InventoryRatio'].median(), inplace=True)

<ipython-input-2-06d702cbf281>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
<ipython-input-2-06d702cbf281>:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].meth

In [3]:
#Convert Quarter to integer
train_df['Quarter'] = train_df['Quarter'].str.extract('(\d+)').astype(int)
test_df['Quarter'] = test_df['Quarter'].str.extract('(\d+)').astype(int)

#Convert Quarter to Month (approximate)
train_df['Month'] = train_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)
test_df['Month'] = test_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)

In [4]:
#Merge economic indicators based on Month
train_merged = train_df.merge(economic_indicators_df, on='Month', how='left')
test_merged = test_df.merge(economic_indicators_df, on='Month', how='left')

#Drop Month column
train_merged.drop(columns=['Month'], inplace=True)
test_merged.drop(columns=['Month'], inplace=True)

In [5]:
#Define categorical columns
categorical_features = ['Company', 'Region', 'Industry', 'Bond rating', 'Stock rating']

#Apply Label Encoding
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    train_merged[col] = le.fit_transform(train_merged[col])
    test_merged[col] = le.transform(test_merged[col])  # Use same encoding for test data
    label_encoders[col] = le

In [6]:
#Interaction Features
train_merged["Revenue_Marketshare"] = train_merged["RevenueGrowth"] * train_merged["Marketshare"]
train_merged["Inventory_Quick"] = train_merged["InventoryRatio"] * train_merged["QuickRatio"]

test_merged["Revenue_Marketshare"] = test_merged["RevenueGrowth"] * test_merged["Marketshare"]
test_merged["Inventory_Quick"] = test_merged["InventoryRatio"] * test_merged["QuickRatio"]

#Rolling Averages for Economic Indicators
rolling_features = ["Consumer Sentiment", "Interest Rate", "PMI", "Money Supply",
                    "NationalEAI", "EastEAI", "WestEAI", "SouthEAI", "NorthEAI"]

for feature in rolling_features:
    train_merged[f"{feature}_rolling"] = train_merged[feature].rolling(window=3, min_periods=1).mean()
    test_merged[f"{feature}_rolling"] = test_merged[feature].rolling(window=3, min_periods=1).mean()

In [7]:
#Define features and target
X = train_merged.drop(columns=['Sales'])
y = train_merged['Sales']

#Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
#Define XGBoost model
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=200, learning_rate=0.05,
                             max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)

#Train the model
xgb_model.fit(X_train, y_train)

#Predict on validation set
y_pred = xgb_model.predict(X_val)

#Evaluate model performance
mae_xgb = mean_absolute_error(y_val, y_pred)
print(f"Baseline XGBoost MAE: {mae_xgb:.2f}")

Baseline XGBoost MAE: 1141.30


In [9]:
#Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

#Initialize RandomizedSearch
random_search = RandomizedSearchCV(xgb.XGBRegressor(objective="reg:squarederror", random_state=42),
                                   param_grid, n_iter=10, cv=3, scoring='neg_mean_absolute_error',
                                   random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

#Best model
best_xgb_model = random_search.best_estimator_

#Predict on validation set with tuned model
y_val_pred_tuned = best_xgb_model.predict(X_val)

#Evaluate performance
tuned_mae_xgb = mean_absolute_error(y_val, y_val_pred_tuned)
print(f"Tuned XGBoost MAE: {tuned_mae_xgb:.2f}")

Tuned XGBoost MAE: 1148.81


In [10]:
#Predict sales for test dataset
test_predictions = best_xgb_model.predict(test_merged.drop(columns=['RowID']))

#Prepare submission file
submission = pd.DataFrame({'RowID': test_df['RowID'], 'Sales': test_predictions})
submission.to_csv("newxgboost_submission.csv", index=False)

print("Submission file saved as 'xgboost_submission.csv'.")

Submission file saved as 'xgboost_submission.csv'.
